In [2]:
import pandas as pd
import os

# Check what got attached
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv


In [3]:
df = pd.read_csv('/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())
print("Unique users:", df['subject'].nunique())


(20400, 34)
  subject  sessionIndex  rep  H.period  DD.period.t  UD.period.t     H.t  \
0    s002             1    1    0.1491       0.3979       0.2488  0.1069   
1    s002             1    2    0.1111       0.3451       0.2340  0.0694   
2    s002             1    3    0.1328       0.2072       0.0744  0.0731   
3    s002             1    4    0.1291       0.2515       0.1224  0.1059   
4    s002             1    5    0.1249       0.2317       0.1068  0.0895   

   DD.t.i  UD.t.i     H.i  ...     H.a  DD.a.n  UD.a.n     H.n  DD.n.l  \
0  0.1674  0.0605  0.1169  ...  0.1349  0.1484  0.0135  0.0932  0.3515   
1  0.1283  0.0589  0.0908  ...  0.1412  0.2558  0.1146  0.1146  0.2642   
2  0.1291  0.0560  0.0821  ...  0.1621  0.2332  0.0711  0.1172  0.2705   
3  0.2495  0.1436  0.1040  ...  0.1457  0.1629  0.0172  0.0866  0.2341   
4  0.1676  0.0781  0.0903  ...  0.1312  0.1582  0.0270  0.0884  0.2517   

   UD.n.l     H.l  DD.l.Return  UD.l.Return  H.Return  
0  0.2583  0.1338       0.3509

In [4]:
import numpy as np

# Select only timing features (drop metadata)
feature_cols = [col for col in df.columns if col not in ['subject', 'sessionIndex', 'rep']]

# Compute ratio features — each H value divided by next H value
# This is the core GhostID innovation — ratios survive speed changes
H_cols = [col for col in feature_cols if col.startswith('H.')]
DD_cols = [col for col in feature_cols if col.startswith('DD.')]
UD_cols = [col for col in feature_cols if col.startswith('UD.')]

print(f"H features: {len(H_cols)}")
print(f"DD features: {len(DD_cols)}")
print(f"UD features: {len(UD_cols)}")
print(f"Total features: {len(feature_cols)}")

# Create ratio features between consecutive H values
for i in range(len(H_cols) - 1):
    df[f'ratio_{H_cols[i]}_{H_cols[i+1]}'] = df[H_cols[i]] / (df[H_cols[i+1]] + 1e-8)

ratio_cols = [col for col in df.columns if col.startswith('ratio_')]
print(f"\nRatio features created: {len(ratio_cols)}")


H features: 11
DD features: 10
UD features: 10
Total features: 31

Ratio features created: 10


In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode user labels
le = LabelEncoder()
df['user_id'] = le.fit_transform(df['subject'])

# Full feature set — raw + ratio
all_features = feature_cols + ratio_cols

# X and y
X = df[all_features].values
y = df['user_id'].values

print(f"Feature vector size: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")
print(f"Number of users: {len(np.unique(y))}")

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}")
print(f"Test: {X_test.shape}")

Feature vector size: 41
Total samples: 20400
Number of users: 51

Train: (16320, 41)
Test: (4080, 41)


In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch version: 2.10.0+cu128
GPU available: True
Device: Tesla T4


In [7]:
# Load data
df = pd.read_csv('/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv')

# Feature engineering
feature_cols = [col for col in df.columns if col not in ['subject', 'sessionIndex', 'rep']]
H_cols = [col for col in feature_cols if col.startswith('H.')]

# Ratio features
for i in range(len(H_cols) - 1):
    df[f'ratio_{H_cols[i]}_{H_cols[i+1]}'] = df[H_cols[i]] / (df[H_cols[i+1]] + 1e-8)

ratio_cols = [col for col in df.columns if col.startswith('ratio_')]
all_features = feature_cols + ratio_cols

# Encode labels
le = LabelEncoder()
df['user_id'] = le.fit_transform(df['subject'])

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(df[all_features].values)
y = df['user_id'].values

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reshape for LSTM — (samples, sequence_length, features)
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print(f"Train shape: {X_train_lstm.shape}")
print(f"Test shape: {X_test_lstm.shape}")
print(f"Classes: {len(np.unique(y))}")

Train shape: (16320, 1, 41)
Test shape: (4080, 1, 41)
Classes: 51


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset
class KeystrokeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).to(device)
        self.y = torch.LongTensor(y).to(device)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# LSTM Model
class GhostIDLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(GhostIDLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.3)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.bn(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# Init
INPUT_SIZE = 41
HIDDEN_SIZE = 256
NUM_LAYERS = 2
NUM_CLASSES = 51
BATCH_SIZE = 64
EPOCHS = 50
LR = 0.001

model = GhostIDLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)

# Dataloaders
train_dataset = KeystrokeDataset(X_train_lstm, y_train)
test_dataset = KeystrokeDataset(X_test_lstm, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

GhostIDLSTM(
  (lstm): LSTM(41, 256, num_layers=2, batch_first=True, dropout=0.3)
  (bn): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=51, bias=True)
)

Total parameters: 872,499


In [9]:
# Training loop
best_acc = 0
train_losses = []
test_accs = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    
    # Evaluate
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    
    acc = 100 * correct / total
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accs.append(acc)
    
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_ghostid_model.pt')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}% | Best: {best_acc:.2f}%")

print(f"\nTraining complete. Best accuracy: {best_acc:.2f}%")

Epoch [5/50] Loss: 0.4562 | Test Acc: 89.83% | Best: 89.83%
Epoch [10/50] Loss: 0.3079 | Test Acc: 92.11% | Best: 92.11%
Epoch [15/50] Loss: 0.1954 | Test Acc: 93.11% | Best: 93.46%
Epoch [20/50] Loss: 0.1659 | Test Acc: 93.55% | Best: 93.55%
Epoch [25/50] Loss: 0.1297 | Test Acc: 93.92% | Best: 93.97%
Epoch [30/50] Loss: 0.1195 | Test Acc: 93.73% | Best: 94.02%
Epoch [35/50] Loss: 0.1016 | Test Acc: 93.80% | Best: 94.09%
Epoch [40/50] Loss: 0.0982 | Test Acc: 93.80% | Best: 94.09%
Epoch [45/50] Loss: 0.0888 | Test Acc: 93.92% | Best: 94.12%
Epoch [50/50] Loss: 0.0853 | Test Acc: 94.29% | Best: 94.29%

Training complete. Best accuracy: 94.29%
